In [1]:
include("dependencies.jl")
include("system_params.jl")
include("functions.jl")

hamitonian1 (generic function with 1 method)

In [2]:
const σ_x, n, Π_g, n, I, σ_minus, σ_plus, σ_z = system_constants()
const Ω, γ_Decay, γ_dephase, V_nn, δ, Δ1_0, T_optimal = unpack_params()

(1.0, 0.0, 0.0, -1000.0, 0.108, 1032.0, 582.0)

In [11]:
function master_eqn(dρ,ρ,p,t)

    n_atoms = 2
    # #2x2 Matrices
    # σ_x = [0 1; 1 0]
    # n = [0 0; 0 1]
    # Π_g = [1 0; 0 0]
    # I = [1 0; 0 1]
    # σ_minus = [0 1; 0 0]
    # σ_plus = [0 0; 1 0]
    # σ_z = [1 0; 0 -1]

    #parameters
    Δ_t, Ω, γ_Decay, γ_dephase, V_nn = p1(t)


    H = hamiltonian1(p1)

    print(H)
end

function solve_master_eqn(p)
    global δ = δ
    tspan = (0.0,T_optimal)
    eqn = ODEProblem(master_eqn, ρ_0, tspan, p)
    sol = solve(eqn, Rodas3(autodiff=false),saveat=1)
    return sol
end

function hamiltonian1(p1::qubit_parameters,t)
    """
    Function to calculate the Hamiltonian for the driving the qubits A-B-C at time t such that
    ti<=t<=tf, where
        - ti=0: start time of the pulse for the qubits A-B-C
        - tf=T1: end time of the pulse for the qubits A-B-C

    Parameters:
        - p:: qubit_parameters
        - t:: Float64

    Returns:
        - H:: Matrix : Hamiltonian at time t
    """
    Δ1_t, Ω, γ_Decay, γ_dephase, V_nn = p1(t)

    σx_a = full_operator(σ_x, 3,[1])
    σx_c = full_operator(σ_x, 3, [3])

    n_a = full_operator(n,3, [1])
    n_c = full_operator(n, 3, [3])

    nn_ab = full_operator(n, 3, [1,2])
    nn_bc = full_operator(n, 3, [2,3])


    H = Ω/2 .* (σx_a + σx_c) + Δ1_t .* (n_a + n_c) + V_nn .* (nn_ab + nn_bc) 

    return H
end

hamiltonian1 (generic function with 1 method)

In [12]:
p = qubit_parameters(Ω, γ_Decay, γ_dephase, V_nn, δ)
t = 1
hamiltonian1(p,t)

8×8 Matrix{Float64}:
 0.0     0.5   0.0   0.0       0.5      0.0    0.0     0.0
 0.5  1031.89  0.0   0.0       0.0      0.5    0.0     0.0
 0.0     0.0   0.0   0.5       0.0      0.0    0.5     0.0
 0.0     0.0   0.5  31.892     0.0      0.0    0.0     0.5
 0.5     0.0   0.0   0.0    1031.89     0.5    0.0     0.0
 0.0     0.5   0.0   0.0       0.5   2063.78   0.0     0.0
 0.0     0.0   0.5   0.0       0.0      0.0   31.892   0.5
 0.0     0.0   0.0   0.5       0.0      0.0    0.5    63.784